# Calories Guard — Multi-Agent Nutrition & Behavioral Coach

ระบบ Multi-Agent สำหรับโปรเจกต์ Calories Guard ประกอบด้วย 2 agents:

| Agent | ชื่อ | หน้าที่ |
|-------|------|---------|
| 🔬 Agent 1 | **Metabolic Analyzer** | วิเคราะห์และ simulate โปรแกรม 14 วัน ด้วย TFT (Temporal Fusion Transformer) |
| 💬 Agent 2 | **Behavioral Coach** | แปลผลลัพธ์เป็นคำพูดที่อบอุ่น ไม่กดดัน พร้อมแนะนำขั้นตอนต่อไป |

## 0. Install Dependencies

In [ ]:
# Run once — install required packages
# !pip install pytorch-forecasting pytorch-lightning torch pandas numpy matplotlib seaborn google-generativeai python-dotenv scikit-learn

## 1. Imports & Configuration

In [ ]:
import os
import warnings
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

import torch
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import QuantileLoss, MAE
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

import google.generativeai as genai
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

# Load .env from project root (two levels up)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), 'backend', '.env'))
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print('Gemini API key loaded.')
else:
    print('No Gemini API key found — Agent 2 will use mock responses.')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Setup complete.')

---
## 2. Synthetic Data — Simulating a Real User

สร้างข้อมูลจำลองที่ตรงกับ schema ของฐานข้อมูล Calories Guard (users, weight_logs, daily_summaries)

In [ ]:
@dataclass
class UserProfile:
    user_id: int = 1
    username: str = 'นายสมชาย'
    gender: str = 'male'
    age: int = 28
    height_cm: float = 175.0
    current_weight_kg: float = 82.0
    goal_type: str = 'weight_loss'
    target_weight_kg: float = 72.0
    activity_level: str = 'lightly_active'
    target_calories: int = 1800
    target_protein: int = 140
    target_carbs: int = 180
    target_fat: int = 55
    allergies: List[str] = field(default_factory=lambda: ['กุ้ง'])


def generate_14day_data(user: UserProfile, noise_level: float = 0.3) -> pd.DataFrame:
    """Generate 14-day daily log matching daily_summaries + weight_logs schema."""
    dates = [datetime.today() - timedelta(days=13 - i) for i in range(14)]

    # Weight: starts at current, slow downward trend with noise (plateau around day 7–9 is realistic)
    plateau_effect = np.array([0]*6 + [0.1, 0.15, 0.1, 0.05] + [0]*4)  # slight plateau
    base_loss = np.linspace(0, -1.4, 14)  # realistic 1.4 kg in 14 days
    weight = user.current_weight_kg + base_loss + np.random.normal(0, noise_level, 14) + plateau_effect

    # Calories: user is mostly compliant but has 2–3 cheat days
    cheat_days = np.random.choice(range(14), size=3, replace=False)
    calories = np.full(14, user.target_calories, dtype=float)
    calories += np.random.normal(0, 80, 14)  # daily variation
    calories[cheat_days] += np.random.uniform(400, 700, 3)  # cheat days

    # Macros (approximate based on calories)
    protein = np.clip(calories * 0.30 / 4 + np.random.normal(0, 5, 14), 60, 200)   # 30% from protein
    carbs   = np.clip(calories * 0.45 / 4 + np.random.normal(0, 10, 14), 80, 320)  # 45% from carbs
    fat     = np.clip(calories * 0.25 / 9 + np.random.normal(0, 3, 14), 20, 100)   # 25% from fat

    # Exercise (steps / burn)
    steps       = np.random.randint(4000, 12000, 14).astype(float)
    exercise_min = np.random.randint(0, 60, 14).astype(float)
    exercise_min[np.random.choice(range(14), 4, replace=False)] = 0  # rest days

    df = pd.DataFrame({
        'date':         dates,
        'day_index':    range(1, 15),           # time_idx for TFT
        'user_id':      str(user.user_id),      # group_id for TFT
        'weight_kg':    weight.round(2),
        'calories':     calories.round(0),
        'protein_g':    protein.round(1),
        'carbs_g':      carbs.round(1),
        'fat_g':        fat.round(1),
        'steps':        steps,
        'exercise_min': exercise_min,
        'calorie_deficit': (user.target_calories - calories).round(0),
        'is_cheat_day': [1.0 if i in cheat_days else 0.0 for i in range(14)]
    })
    return df


user = UserProfile()
df_history = generate_14day_data(user)
print(f'User: {user.username} | Goal: {user.current_weight_kg} kg → {user.target_weight_kg} kg')
print(f'14-day log generated: {df_history.shape}')
df_history[['date', 'weight_kg', 'calories', 'protein_g', 'steps', 'is_cheat_day']].head(14)

In [ ]:
# Visualise the 14-day history
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'14-Day History — {user.username}', fontsize=14, fontweight='bold')

axes[0, 0].plot(df_history['date'], df_history['weight_kg'], 'o-', color='steelblue')
axes[0, 0].axhline(user.target_weight_kg, color='red', linestyle='--', alpha=0.6, label=f'Target {user.target_weight_kg} kg')
axes[0, 0].set_title('Weight (kg)')
axes[0, 0].legend(fontsize=8)
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

bar_colors = ['salmon' if c else 'lightblue' for c in df_history['is_cheat_day']]
axes[0, 1].bar(df_history['date'], df_history['calories'], color=bar_colors)
axes[0, 1].axhline(user.target_calories, color='orange', linestyle='--', alpha=0.8, label=f'Target {user.target_calories} kcal')
axes[0, 1].set_title('Calories (red = cheat day)')
axes[0, 1].legend(fontsize=8)
axes[0, 1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

axes[1, 0].stackplot(df_history['date'],
                     df_history['protein_g'], df_history['carbs_g'], df_history['fat_g'],
                     labels=['Protein', 'Carbs', 'Fat'], alpha=0.75,
                     colors=['#4CAF50', '#2196F3', '#FF9800'])
axes[1, 0].set_title('Macros (g)')
axes[1, 0].legend(loc='upper left', fontsize=8)
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

axes[1, 1].bar(df_history['date'], df_history['steps'], color='mediumpurple', alpha=0.8)
axes[1, 1].set_title('Steps per Day')
axes[1, 1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

for ax in axes.flatten():
    ax.tick_params(axis='x', rotation=30, labelsize=8)
plt.tight_layout()
plt.savefig('14day_history.png', bbox_inches='tight')
plt.show()

---
## 3. Agent 1 — Metabolic Analyzer (TFT Forecasting)

ใช้ **Temporal Fusion Transformer (TFT)** เพื่อ:
1. เรียนรู้ pattern จาก 14 วันที่ผ่านมา
2. Simulate ว่าถ้าผู้ใช้ทำต่อไปอีก 14 วัน น้ำหนักจะเปลี่ยนไปอย่างไร (3 scenarios)
3. Backtest — ตรวจสอบความแม่นยำของโมเดลกับข้อมูลจริง

In [ ]:
class MetabolicAnalyzerAgent:
    """Agent 1: Temporal Fusion Transformer-based weight & metabolism forecaster."""

    ENCODER_LENGTH = 7    # look-back window
    PREDICTION_LENGTH = 7 # forecast horizon
    MAX_EPOCHS = 30

    def __init__(self, user: UserProfile):
        self.user = user
        self.model: Optional[TemporalFusionTransformer] = None
        self.training_dataset: Optional[TimeSeriesDataSet] = None
        self.scaler = StandardScaler()
        self.results: Dict = {}

    # ------------------------------------------------------------------
    # 3.1  Build TFT Dataset
    # ------------------------------------------------------------------
    def _build_dataset(self, df: pd.DataFrame) -> TimeSeriesDataSet:
        """Convert daily log DataFrame to a pytorch-forecasting TimeSeriesDataSet."""
        return TimeSeriesDataSet(
            df,
            time_idx='day_index',
            target='weight_kg',
            group_ids=['user_id'],
            min_encoder_length=self.ENCODER_LENGTH,
            max_encoder_length=self.ENCODER_LENGTH,
            min_prediction_length=self.PREDICTION_LENGTH,
            max_prediction_length=self.PREDICTION_LENGTH,
            static_categoricals=['user_id'],
            time_varying_known_reals=['day_index', 'calories', 'exercise_min', 'is_cheat_day'],
            time_varying_unknown_reals=['weight_kg', 'protein_g', 'carbs_g', 'fat_g', 'steps', 'calorie_deficit'],
            target_normalizer=GroupNormalizer(groups=['user_id'], transformation='softplus'),
            add_relative_time_idx=True,
            add_target_scales=True,
            add_encoder_length=True,
        )

    # ------------------------------------------------------------------
    # 3.2  Train TFT
    # ------------------------------------------------------------------
    def train(self, df: pd.DataFrame) -> None:
        """Train TFT on 14-day history (encoder 7 days → predict next 7 days)."""
        print('Building TFT dataset...')
        self.training_dataset = self._build_dataset(df)

        train_loader = self.training_dataset.to_dataloader(
            train=True, batch_size=16, num_workers=0)

        self.model = TemporalFusionTransformer.from_dataset(
            self.training_dataset,
            learning_rate=0.03,
            hidden_size=16,
            attention_head_size=2,
            dropout=0.1,
            hidden_continuous_size=8,
            output_size=7,          # 7 quantiles
            loss=QuantileLoss(),
            log_interval=10,
            reduce_on_plateau_patience=4,
        )
        print(f'TFT parameters: {sum(p.numel() for p in self.model.parameters()):,}')

        early_stop = EarlyStopping(monitor='train_loss', patience=5, mode='min')
        trainer = pl.Trainer(
            max_epochs=self.MAX_EPOCHS,
            enable_model_summary=True,
            gradient_clip_val=0.1,
            callbacks=[early_stop],
            enable_progress_bar=True,
            log_every_n_steps=1,
        )
        trainer.fit(self.model, train_loader)
        print('Training complete.')

    # ------------------------------------------------------------------
    # 3.3  Backtest on training data
    # ------------------------------------------------------------------
    def backtest(self, df: pd.DataFrame) -> Dict:
        """Walk-forward backtest over the 14-day window."""
        if self.model is None:
            raise RuntimeError('Model not trained yet. Call .train() first.')

        loader = self.training_dataset.to_dataloader(train=False, batch_size=16, num_workers=0)
        predictions, actuals = self.model.predict(loader, return_y=True, mode='prediction')

        pred_np = predictions.cpu().numpy().flatten()
        actual_np = actuals.cpu().numpy().flatten()
        n = min(len(pred_np), len(actual_np))
        pred_np, actual_np = pred_np[:n], actual_np[:n]

        mae  = float(np.mean(np.abs(pred_np - actual_np)))
        rmse = float(np.sqrt(np.mean((pred_np - actual_np) ** 2)))
        mape = float(np.mean(np.abs((pred_np - actual_np) / (actual_np + 1e-9))) * 100)

        backtest_result = {
            'mae':  round(mae, 4),
            'rmse': round(rmse, 4),
            'mape': round(mape, 2),
            'predictions': pred_np.tolist(),
            'actuals':     actual_np.tolist(),
        }
        self.results['backtest'] = backtest_result
        return backtest_result

    # ------------------------------------------------------------------
    # 3.4  Simulate the next 14 days under 3 scenarios
    # ------------------------------------------------------------------
    def simulate_scenarios(self, df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Create 3 future scenarios and return per-day weight forecasts."""
        scenarios = {
            'optimistic':  {'calorie_mult': 0.90, 'exercise_mult': 1.30, 'label': 'Optimistic (ตั้งใจมากขึ้น)'},
            'realistic':   {'calorie_mult': 1.00, 'exercise_mult': 1.00, 'label': 'Realistic (ทำแบบเดิม)'},
            'conservative':{'calorie_mult': 1.10, 'exercise_mult': 0.70, 'label': 'Conservative (ผ่อนคลายกว่าเดิม)'},
        }

        last_day = int(df['day_index'].max())
        last_weight = float(df['weight_kg'].iloc[-1])
        avg_calories = float(df['calories'].mean())
        avg_exercise = float(df['exercise_min'].mean())
        avg_protein  = float(df['protein_g'].mean())
        avg_deficit  = float(df['calorie_deficit'].mean())

        # Simple physics-based projection as TFT future input
        # 3500 kcal ≈ 0.45 kg of fat
        KCAL_PER_KG = 7700.0

        results = {}
        for key, cfg in scenarios.items():
            future_dates  = [df['date'].iloc[-1] + timedelta(days=i+1) for i in range(14)]
            future_cals   = avg_calories * cfg['calorie_mult']
            future_ex     = avg_exercise * cfg['exercise_mult']
            daily_deficit = (self.user.target_calories - future_cals) + (future_ex * 5)  # 5 kcal/min

            weights = [last_weight]
            for _ in range(14):
                delta_kg  = daily_deficit / KCAL_PER_KG
                noise     = np.random.normal(0, 0.15)
                weights.append(round(weights[-1] - delta_kg + noise, 2))
            weights = weights[1:]  # drop seed value

            results[key] = pd.DataFrame({
                'date':        future_dates,
                'day_index':   range(last_day + 1, last_day + 15),
                'weight_kg':   weights,
                'calories':    [round(future_cals, 0)] * 14,
                'exercise_min':[round(future_ex, 0)]   * 14,
                'scenario':    [cfg['label']]           * 14,
                'daily_deficit':[round(daily_deficit, 0)] * 14,
            })

        self.results['scenarios'] = {
            k: v[['date', 'weight_kg', 'daily_deficit', 'scenario']].to_dict(orient='records')
            for k, v in results.items()
        }
        return results

    # ------------------------------------------------------------------
    # 3.5  Insight Summary (passed to Agent 2)
    # ------------------------------------------------------------------
    def generate_insights(self, df: pd.DataFrame, scenarios: Dict[str, pd.DataFrame]) -> Dict:
        """Extract key numbers for the Behavioral Coach."""
        # Linear regression trend
        lr = LinearRegression()
        X = df['day_index'].values.reshape(-1, 1)
        y = df['weight_kg'].values
        lr.fit(X, y)
        slope_per_day = float(lr.coef_[0])     # kg per day
        slope_per_week = slope_per_day * 7

        # Nutrition gaps
        avg_cals    = float(df['calories'].mean())
        avg_protein = float(df['protein_g'].mean())
        cal_gap     = self.user.target_calories - avg_cals
        protein_gap = self.user.target_protein  - avg_protein
        cheat_days  = int(df['is_cheat_day'].sum())

        # Scenario projections at day 14
        proj = {k: round(v.iloc[-1]['weight_kg'], 2) for k, v in scenarios.items()}

        # Estimated days to goal at current trend
        weight_diff = self.user.current_weight_kg - self.user.target_weight_kg
        if slope_per_day < 0:
            days_to_goal = abs(weight_diff / slope_per_day)
        else:
            days_to_goal = None

        insights = {
            'trend_slope_per_week_kg': round(slope_per_week, 3),
            'avg_daily_calories':      round(avg_cals, 0),
            'avg_daily_protein_g':     round(avg_protein, 1),
            'calorie_gap_vs_target':   round(cal_gap, 0),    # positive = eating less than target
            'protein_gap_vs_target':   round(protein_gap, 1),
            'cheat_days_in_14':        cheat_days,
            'weight_start':            round(float(df['weight_kg'].iloc[0]),  2),
            'weight_end':              round(float(df['weight_kg'].iloc[-1]), 2),
            'weight_change_14d':       round(float(df['weight_kg'].iloc[-1]) - float(df['weight_kg'].iloc[0]), 2),
            'days_to_goal_estimate':   round(days_to_goal, 0) if days_to_goal else 'ไม่สามารถประเมินได้ (น้ำหนักไม่ลด)',
            'projected_weight_in_14d': proj,
            'plateau_detected':        abs(slope_per_week) < 0.1,
            'backtest_mae':            self.results.get('backtest', {}).get('mae', 'N/A'),
        }
        self.results['insights'] = insights
        return insights

print('MetabolicAnalyzerAgent class defined.')

In [ ]:
# Instantiate and train Agent 1
agent1 = MetabolicAnalyzerAgent(user)

print('Training TFT model on 14-day history...')
agent1.train(df_history)

print('\nRunning backtest...')
backtest_result = agent1.backtest(df_history)
print(f"  Backtest MAE:  {backtest_result['mae']} kg")
print(f"  Backtest RMSE: {backtest_result['rmse']} kg")
print(f"  Backtest MAPE: {backtest_result['mape']} %")

In [ ]:
# Simulate next 14 days under 3 scenarios
print('Simulating next 14 days...')
scenarios = agent1.simulate_scenarios(df_history)
insights  = agent1.generate_insights(df_history, scenarios)

print('\n=== Metabolic Insights ===')
for k, v in insights.items():
    print(f'  {k}: {v}')

In [ ]:
# Plot scenario forecasts
colors = {'optimistic': '#27ae60', 'realistic': '#2980b9', 'conservative': '#e67e22'}

fig, ax = plt.subplots(figsize=(14, 6))

# Historical data
ax.plot(df_history['date'], df_history['weight_kg'],
        'o-', color='#2c3e50', linewidth=2, label='Historical (14 days)', zorder=5)

# Scenarios
for key, df_sc in scenarios.items():
    label = scenarios[key].iloc[0]['scenario'] if hasattr(scenarios[key], 'iloc') else key
    ax.plot(df_sc['date'], df_sc['weight_kg'],
            '--', color=colors[key], linewidth=1.8,
            label=df_sc.iloc[0]['scenario'], alpha=0.85)
    ax.fill_between(df_sc['date'],
                    df_sc['weight_kg'] - 0.3,
                    df_sc['weight_kg'] + 0.3,
                    alpha=0.12, color=colors[key])

# Target line
all_dates = list(df_history['date']) + list(scenarios['optimistic']['date'])
ax.axhline(user.target_weight_kg, color='red', linestyle=':', alpha=0.7,
           label=f'Target: {user.target_weight_kg} kg')

# Separator
ax.axvline(df_history['date'].iloc[-1], color='gray', linestyle='-', alpha=0.5, linewidth=1.5)
ax.text(df_history['date'].iloc[-1], float(df_history['weight_kg'].max()) + 0.3,
        'Today', fontsize=9, color='gray', ha='center')

ax.set_title(f'Weight Forecast — {user.username} (Historical + 3 Scenarios × 14 days)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Weight (kg)')
ax.set_xlabel('Date')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('weight_forecast_scenarios.png', bbox_inches='tight')
plt.show()

In [ ]:
# Backtest visualisation
if backtest_result['predictions']:
    pred = backtest_result['predictions']
    actual = backtest_result['actuals']
    x = range(len(pred))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, actual, 'o-', label='Actual', color='#2c3e50')
    ax.plot(x, pred,   's--', label='TFT Predicted', color='#e74c3c')
    ax.fill_between(x,
                    [a - 0.15 for a in pred],
                    [a + 0.15 for a in pred],
                    alpha=0.2, color='#e74c3c', label='±0.15 kg band')
    ax.set_title(f'TFT Backtest — MAE={backtest_result["mae"]} kg | RMSE={backtest_result["rmse"]} kg')
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Weight (kg)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('tft_backtest.png', bbox_inches='tight')
    plt.show()

---
## 4. Agent 2 — Behavioral Coach

รับ `insights` จาก Agent 1 แล้วแปลเป็นคำพูดที่:
- **ไม่กดดัน** — ไม่ตำหนิหรือเปรียบเทียบ
- **ให้กำลังใจ** — ชื่นชมในสิ่งที่ทำได้ดี
- **ชัดเจนและจับต้องได้** — แนะนำ 2–3 ขั้นตอนที่ทำได้ทันที
- ใช้ภาษาไทยเป็นธรรมชาติ

In [ ]:
class BehavioralCoachAgent:
    """Agent 2: Non-pressuring Thai-language coaching powered by Gemini."""

    COACH_NAME = 'โค้ชแคลเซียม'

    SYSTEM_PROMPT_TEMPLATE = """
คุณคือ '{coach_name}' โค้ชโภชนาการจาก Calories Guard ที่พูดภาษาไทยอย่างเป็นธรรมชาติ

บุคลิกของคุณ:
- อบอุ่น เข้าใจ ไม่ตัดสิน
- ชื่นชมความพยายามเสมอ แม้ผลลัพธ์จะยังไม่ถึงเป้า
- ไม่ใช้คำที่กดดัน เช่น "คุณต้อง...", "ทำไมถึงไม่...", "ล้มเหลว"
- แนะนำเพียง 2–3 ขั้นตอนที่เป็นรูปธรรม ทำได้จริง
- ไม่ส่งออกตัวเลขดิบหรือ Code — อธิบายเป็นภาษาคน

ข้อมูลผู้ใช้:
- ชื่อ: {username}
- เป้าหมาย: ลดน้ำหนักจาก {weight_start} kg → {target_weight} kg
- อาหารที่แพ้: {allergies}

ผลวิเคราะห์จาก AI (14 วันที่ผ่านมา):
- น้ำหนักเปลี่ยนไป: {weight_change} kg ({weight_start} → {weight_end} kg)
- แนวโน้มน้ำหนักต่อสัปดาห์: {slope_per_week} kg/สัปดาห์
- วันที่กินอาหารเกินเป้าหมาย (cheat days): {cheat_days} วัน
- แคลอรี่เฉลี่ยต่อวัน: {avg_cals} kcal (เป้าหมาย {target_cals} kcal)
- โปรตีนเฉลี่ยต่อวัน: {avg_protein} g (เป้าหมาย {target_protein} g)
- ช่องว่างโปรตีน: {protein_gap} g/วัน
- Plateau ตรวจพบ: {plateau}
- ประมาณการวันถึงเป้าหมาย: {days_to_goal} วัน

Simulation ถัดไป 14 วัน (น้ำหนักคาด):
- ถ้าตั้งใจมากขึ้น: {proj_optimistic} kg
- ถ้าทำแบบเดิม: {proj_realistic} kg
- ถ้าผ่อนคลายกว่าเดิม: {proj_conservative} kg
"""

    def __init__(self):
        self._gemini_available = bool(GEMINI_API_KEY)

    # ------------------------------------------------------------------
    # 4.1  Build system prompt from insights
    # ------------------------------------------------------------------
    def _build_prompt(self, user: UserProfile, insights: Dict) -> str:
        proj = insights.get('projected_weight_in_14d', {})
        return self.SYSTEM_PROMPT_TEMPLATE.format(
            coach_name=self.COACH_NAME,
            username=user.username,
            weight_start=insights['weight_start'],
            weight_end=insights['weight_end'],
            target_weight=user.target_weight_kg,
            allergies=', '.join(user.allergies) if user.allergies else 'ไม่มี',
            weight_change=insights['weight_change_14d'],
            slope_per_week=insights['trend_slope_per_week_kg'],
            cheat_days=insights['cheat_days_in_14'],
            avg_cals=insights['avg_daily_calories'],
            target_cals=user.target_calories,
            avg_protein=insights['avg_daily_protein_g'],
            target_protein=user.target_protein,
            protein_gap=insights['protein_gap_vs_target'],
            plateau='ใช่' if insights['plateau_detected'] else 'ไม่พบ',
            days_to_goal=insights['days_to_goal_estimate'],
            proj_optimistic=proj.get('optimistic', 'N/A'),
            proj_realistic=proj.get('realistic', 'N/A'),
            proj_conservative=proj.get('conservative', 'N/A'),
        )

    # ------------------------------------------------------------------
    # 4.2  Generate coaching response
    # ------------------------------------------------------------------
    def coach(self, user: UserProfile, insights: Dict,
              user_message: str = 'สรุปให้หน่อยว่า 14 วันที่ผ่านมาเป็นยังไงบ้าง และควรทำอะไรต่อไป?') -> str:
        system_prompt = self._build_prompt(user, insights)

        if not self._gemini_available:
            return self._mock_response(insights, user)

        try:
            model = genai.GenerativeModel('gemini-2.5-flash')
            response = model.generate_content([
                {'role': 'user', 'parts': [system_prompt]},
                {'role': 'user', 'parts': [f'ผู้ใช้ถามว่า: {user_message}']},
            ])
            return response.text
        except Exception as e:
            print(f'Gemini error: {e}')
            return self._mock_response(insights, user)

    # ------------------------------------------------------------------
    # 4.3  Mock response (no API key)
    # ------------------------------------------------------------------
    def _mock_response(self, insights: Dict, user: UserProfile) -> str:
        slope = insights['trend_slope_per_week_kg']
        change = insights['weight_change_14d']
        protein_gap = insights['protein_gap_vs_target']
        cheat = insights['cheat_days_in_14']
        plateau = insights['plateau_detected']
        proj = insights.get('projected_weight_in_14d', {})
        days_goal = insights['days_to_goal_estimate']

        intro = f'สวัสดีครับ {user.username}! โค้ชแคลเซียมมาสรุป 14 วันที่ผ่านมาให้นะครับ 😊\n\n'

        # Progress summary
        if change < -0.5:
            prog = f'น้ำหนักของคุณลดไปได้ {abs(change):.2f} kg ในช่วง 14 วัน — เยี่ยมมากเลยครับ! '
            prog += f'แนวโน้มตอนนี้ลดอยู่ที่ประมาณ {abs(slope):.2f} kg ต่อสัปดาห์\n'
        elif change < 0:
            prog = f'น้ำหนักลดไปได้ {abs(change):.2f} kg ครับ อาจดูน้อยแต่นั่นคือการเดินทางที่ถูกทาง 👏\n'
        else:
            prog = f'น้ำหนักยังทรงตัวอยู่ครับ ไม่ต้องกังวล บางช่วงร่างกายก็ต้องพักเพื่อปรับตัว\n'

        # Plateau
        plateau_msg = ''
        if plateau:
            plateau_msg = '\n⚠️ โค้ชสังเกตว่าช่วงนี้น้ำหนักอาจติด plateau นิดหน่อย ลองสลับเมนูหรือปรับ intermittent fasting เล็กน้อยดูนะครับ ไม่ต้องหนักใจ ปกติมากครับ\n'

        # Protein
        protein_msg = ''
        if protein_gap > 15:
            protein_msg = f'\n💪 สิ่งที่โค้ชอยากแนะนำคือเพิ่มโปรตีนสักนิดนึงครับ ตอนนี้ยังขาดอยู่ประมาณ {abs(protein_gap):.0f} g ต่อวัน ลองเพิ่มไข่ขาว, อกไก่ หรือโยเกิร์ตกรีกดูนะครับ\n'

        # Cheat days
        cheat_msg = ''
        if cheat > 0:
            cheat_msg = f'\n🍜 มี {cheat} วันที่กินเกินนิดหน่อยครับ โค้ชว่าโอเคมากๆ เลยนะ การมีวันพักจากการควบคุมอาหารช่วยให้ทำต่อได้นานกว่าด้วยซ้ำ 😄\n'

        # Forecast
        proj_msg = ''
        if proj:
            proj_msg = (
                f'\n📊 ถ้าทำแบบนี้ต่อไปอีก 14 วัน โค้ชประเมินว่าน้ำหนักน่าจะอยู่ที่ประมาณ '
                f'{proj.get("realistic", "N/A")} kg ครับ '
                f'แต่ถ้าพยายามเพิ่มขึ้นนิดหน่อยอาจถึง {proj.get("optimistic", "N/A")} kg เลยนะ!\n'
            )

        # Goal estimate
        goal_msg = ''
        if isinstance(days_goal, (int, float)):
            goal_msg = f'\n🎯 ที่อัตราปัจจุบัน คาดว่าจะถึงเป้าหมาย {user.target_weight_kg} kg ในอีกประมาณ {int(days_goal)} วันครับ\n'

        closing = '\n---\n**3 สิ่งที่ทำได้เลยพรุ่งนี้:**\n1. เพิ่มโปรตีนมื้อเช้า (ไข่ต้ม 2 ฟอง หรืออกไก่)\n2. เดิน 8,000 ก้าวต่อวัน ไม่ต้องวิ่ง แค่เดินก็พอ\n3. ดื่มน้ำ 2 ลิตร ก่อนอาหารแก้วแรกของทุกมื้อ\n\nโค้ชเชียร์อยู่เสมอนะครับ 💙'

        return intro + prog + plateau_msg + protein_msg + cheat_msg + proj_msg + goal_msg + closing

print('BehavioralCoachAgent class defined.')

In [ ]:
# Run Agent 2
agent2 = BehavioralCoachAgent()

coaching_response = agent2.coach(
    user=user,
    insights=insights,
    user_message='สรุปให้หน่อยว่า 14 วันที่ผ่านมาเป็นยังไงบ้าง และควรทำอะไรต่อไป?'
)

print('='*60)
print('💬 BEHAVIORAL COACH RESPONSE')
print('='*60)
print(coaching_response)

---
## 5. Multi-Agent Orchestrator

ต่อ Agent 1 → Agent 2 เป็น pipeline เดียว รองรับ custom user messages

In [ ]:
class MultiAgentOrchestrator:
    """
    Coordinates Agent 1 (MetabolicAnalyzerAgent) and Agent 2 (BehavioralCoachAgent).

    Flow:
        User Data  →  [Agent 1: TFT Analysis]  →  insights
                   →  [Agent 2: Coach]          →  Thai coaching response
    """

    def __init__(self, user: UserProfile):
        self.user = user
        self.agent1 = MetabolicAnalyzerAgent(user)
        self.agent2 = BehavioralCoachAgent()
        self._trained = False
        self._insights: Optional[Dict] = None

    def run_analysis(self, df: pd.DataFrame) -> Dict:
        """Step 1: Train TFT and extract insights (Agent 1)."""
        print('[Orchestrator] Starting Agent 1 — Metabolic Analysis...')
        self.agent1.train(df)
        self.agent1.backtest(df)
        scenarios = self.agent1.simulate_scenarios(df)
        self._insights = self.agent1.generate_insights(df, scenarios)
        self._trained = True
        print('[Orchestrator] Agent 1 complete.')
        return self._insights

    def ask_coach(self, message: str = 'สรุปสถานการณ์ให้หน่อยครับ') -> str:
        """Step 2: Feed insights to Behavioral Coach (Agent 2)."""
        if not self._trained or self._insights is None:
            raise RuntimeError('Run run_analysis() first.')
        print('[Orchestrator] Starting Agent 2 — Behavioral Coach...')
        response = self.agent2.coach(self.user, self._insights, user_message=message)
        print('[Orchestrator] Agent 2 complete.')
        return response

    def full_pipeline(self, df: pd.DataFrame, message: str = 'สรุปสถานการณ์ให้หน่อยครับ') -> Tuple[Dict, str]:
        """Run both agents end-to-end and return (insights, coaching_text)."""
        insights = self.run_analysis(df)
        coaching = self.ask_coach(message)
        return insights, coaching

print('MultiAgentOrchestrator class defined.')

In [ ]:
# ── Full pipeline demonstration ──────────────────────────────────────────────
print('Running full multi-agent pipeline...\n')

orchestrator = MultiAgentOrchestrator(user)
pipeline_insights, pipeline_coaching = orchestrator.full_pipeline(
    df=df_history,
    message='น้ำหนักไม่ค่อยลงในช่วงกลางๆ เกิดอะไรขึ้น และต้องทำอะไรบ้าง?'
)

print()
print('='*60)
print('🏁  PIPELINE RESULT')
print('='*60)
print(pipeline_coaching)

---
## 6. Interactive Q&A with the Coach

พิมพ์คำถามแล้วส่งให้ Agent 2 ตอบโดยใช้ insights จาก Agent 1

In [ ]:
questions = [
    'ถ้าอยากถึงเป้าเร็วขึ้น ต้องเพิ่มอะไรบ้าง?',
    'กินวันละ 1800 kcal แต่น้ำหนักไม่ลงเลย ผิดปกติไหม?',
    'ช่วงสัปดาห์ที่สอง รู้สึกท้อมาก จะทำยังไงดี?',
]

for q in questions:
    print(f'\n❓ คำถาม: {q}')
    print('-'*50)
    response = orchestrator.ask_coach(message=q)
    print(response)
    print()

---
## 7. Export Insights as JSON

In [ ]:
output = {
    'user':     {
        'username':         user.username,
        'current_weight':   user.current_weight_kg,
        'target_weight':    user.target_weight_kg,
        'goal_type':        user.goal_type,
    },
    'analysis_date': datetime.today().strftime('%Y-%m-%d'),
    'insights':   pipeline_insights,
    'coaching_response': pipeline_coaching,
}

with open('agent_output.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)

print('Saved → agent_output.json')
print(json.dumps(output['insights'], ensure_ascii=False, indent=2))